# Fine-tune Whisper on CDLI Non-Standard Speech Datasets



In [1]:
from huggingface_hub import login
HF_TOKEN = input()
login(token=HF_TOKEN)

## Settings 

--> adapt for your scenario

### Directories

In [2]:
import os 

# storage in Volume that will persist
LOCAL_STORAGE_DIR = '/jupyter_kernel'

BASE_DIR = os.path.join(LOCAL_STORAGE_DIR, 'trained_models')
!mkdir -p {BASE_DIR}

# directory for model training
OUTPUT_DIR = os.path.join(BASE_DIR, 'whisper-small-kenyan-english-nonstandard_v1_run1')
# OUTPUT_DIR = os.path.join(BASE_DIR, 'sw_nonstandard_tune_whisper_small_run3')

print(f"Will write model to: {OUTPUT_DIR}")
if os.path.exists(OUTPUT_DIR):
    raise ValueError(f"Output directory already exists - if you continue this will overwrite data and may lead to strange results...")


Will write model to: /jupyter_kernel/trained_models/whisper-small-kenyan-english-nonstandard_v1_run1


ValueError: Output directory already exists - if you continue this will overwrite data and may lead to strange results...

### Model and Dataset settings

In [3]:
# WHISPER_MODEL_TYPE = "openai/whisper-tiny" 
WHISPER_MODEL_TYPE = "openai/whisper-small" 
# WHISPER_MODEL_TYPE = "openai/whisper-large-v3" 

LANGUAGE = 'en'
DATASET_NAME = "cdli/kenyan_english_nonstandard_speech_v1.0"

# LANGUAGE = 'sw'
# DATASET_NAME = "cdli/kenyan_swahili_nonstandard_speech_v0.9"


In [4]:
# which parts of the model to update
UPDATE_ENCODER = True
UPDATE_PROJ = True
UPDATE_DECODER = True

# Turn on SpecAugment
USE_SPECAUGMENT = True

In [5]:

#######################
## don't change these!
######################


TASK = "transcribe"

BASE_MODEL_NAME = WHISPER_MODEL_TYPE
print('Base model will be loaded from:', BASE_MODEL_NAME)

Base model will be loaded from: openai/whisper-small


### Trainer Settings

--> adjust as needed or keep defaults (these settings should be a good starting point)

In [6]:
LOGGING_STEPS = 5
# if save steps is 0, only last and best model will be written
SAVE_STEPS = 50

# training duration
MAX_EPOCHS = 10
MAX_STEPS = 2000  # for larger datasets, you will want to increase this

# Learning Rate and LR Scheduler (LR_END and LR_DECAY_POWER only apply to polynomial)
LEARNING_RATE = 3e-6 #@param
LR_SCHEDULER_TYPE = 'polynomial' # constant_with_warmup or polynomial
LR_WARMUP_STEPS = 100
LR_END = 1e-8
LR_DECAY_POWER = 1
# see: https://huggingface.co/docs/transformers/v4.46.2/en/main_classes/optimizer_schedules#transformers.SchedulerType
# and here: https://www.kaggle.com/code/snnclsr/learning-rate-schedulers
# constant --> 'constant_with_warmup'
# polynomial --> 'get_polynomial_decay_schedule_with_warmup'

BATCH_SIZE = 32
EVAL_BATCH_SIZE = 16

#@markdown other settings relevant for evaluation
MAX_GEN_LEN = 128 # increase if your data has long sequences!
EVAL_ON_START = True
EVAL_STEPS = 50

# for CPU, set both to false
USE_FP16 = True
USE_BF16 = False # only some GPUs support this, eg A100, A40

# checkpoints get huge for large models (~18 GB!)
NUM_CHECKPOINTS_TO_STORE = 2

## Imports and Prep

--> no need to change anything here, just run

In [7]:
import datasets
from huggingface_hub import hf_hub_download
import numpy as np
import pandas as pd
import os
import torch

# more efficient dataset handling
datasets.disable_caching()
print('cache:', datasets.is_caching_enabled())

torch.set_num_threads(1)
torch.get_num_threads()

# check if we have gpu
if torch.cuda.is_available():
    print("GPU is available")
else:
    print("GPU is not available, using CPU instead")

cache: False
GPU is available


In [8]:
from huggingface_hub import hf_hub_download

import random
import torchaudio
import librosa


import tarfile
import datasets
import matplotlib.pyplot as plt
import pandas as pd

import torch
import time


from dataclasses import dataclass
from typing import Any, Dict, List, Union

from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer

from transformers import WhisperProcessor
from transformers import WhisperForConditionalGeneration
import os
import csv
import shutil
import numpy as np


import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

transcript_normalizer = BasicTextNormalizer()

In [9]:
def count_trainable_parameters(model):
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

In [10]:
def get_wer(references, predictions, normalize=True, verbose=True):
  rs = references
  ps = predictions
  if normalize:
    ps = [transcript_normalizer(x) for x in predictions]
    rs = [transcript_normalizer(x) for x in references]
  if verbose:
    for r, p in zip(rs, ps):
      print(r)
      print(p)
      print()

  return wer_metric.compute(references=rs, predictions=ps)


def compute_metrics(pred):
    # for training metrics
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_strs = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_strs = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # calculate a per-example average
    wers = []
    cers = []
    for pred_str, label_str in zip(pred_strs, label_strs):
      p = transcript_normalizer(pred_str)
      l = transcript_normalizer(label_str)
      wer = wer_metric.compute(predictions=[p], references=[l])
      cer = cer_metric.compute(predictions=[p], references=[l])
      wers.append(wer)
      cers.append(cer)

    wer = np.mean([min(1.0,x) for x in wers])
    cer = np.mean([min(1.0,x) for x in cers])
    print('adjusted:', wer, cer)
    print('un-adjusted:', np.mean(wers), np.mean(cers))
    return {"wer": wer, "cer": cer}



In [11]:
def load_dataset(dataset_name, split='test', limit_to_30_seconds=True):
    """
    Load a dataset from Hugging Face Hub.
    If limit_to_30_seconds is True, will only load examples with audio length <= 30 seconds.
    """
    if split not in ['train', 'test', 'validation']:
        raise ValueError("split must be one of 'train', 'test', or 'validation'")
    ds = datasets.load_dataset(dataset_name, split=split, streaming=False)
    orig_len = len(ds)
    if limit_to_30_seconds:
        ds = ds.filter(lambda example: example['audio_length'] <= 30)
        print(f"Filtered dataset from {orig_len} to {len(ds)} examples with audio length <= 30 seconds")
    return ds

In [12]:
# The following warning can be ignored:
# "The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results."
# See: https://discuss.huggingface.co/t/finetuning-whisper-attention-mask-not-set-and-canot-be-inferred/97456
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

## Download datasets and prepare features

--> no need to change anything here, just run

### Optimizing some settings for dataset access

In [13]:
datasets.disable_caching()
print('cache:', datasets.is_caching_enabled())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device is: ', device)

# IMPORTANT! need to set to 1 to avoid the mapping to hang!
torch.set_num_threads(1)
torch.get_num_threads()

num_proc = min(32, os.cpu_count())
print('# processors:', num_proc)



cache: False
device is:  cuda
# processors: 20


### Load feature extractor

--> for the model type you specified above

In [14]:

# Load processor
print('Using Language: ', LANGUAGE)
print('Using model:', WHISPER_MODEL_TYPE)
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_TYPE, language=LANGUAGE, task=TASK)

# since this tokenizer isn't a FastTokenizer, so there is no point in running it with is_batched=True
# see: processor.tokenizer.is_fast
def prepare_features(example):
    example["input_features"] = processor.feature_extractor(example["audio"]["array"], sampling_rate=example["audio"]["sampling_rate"]).input_features[0]
    example["labels"] = processor.tokenizer(example["transcription"]).input_ids
    # also count number of tokens
    example["token_length"] = len(example["labels"])
    return example

Using Language:  en
Using model: openai/whisper-small


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

### Load non-standard speech dataset

We need to filter to 30 seconds, as Whisper can only train on that.

In [15]:
# Filter function (define once, use for all splits)
def is_valid_label_length(example):
    if 'labels' not in example:
        return False
    # Count actual label tokens (excluding padding token -100)
    valid_labels = [l for l in example['labels'] if l != -100]
    return len(valid_labels) <= 448

In [16]:
train_dataset = load_dataset(DATASET_NAME, split='train', limit_to_30_seconds=True)
train_dataset = train_dataset.map(prepare_features, remove_columns=['audio'], writer_batch_size=1, num_proc=num_proc)
print(f"Loaded TRAIN dataset with {len(train_dataset)} examples")

Generating test split:   0%|          | 0/928 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/542 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/4378 [00:00<?, ? examples/s]

Generating extra split:   0%|          | 0/150 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4378 [00:00<?, ? examples/s]

Filtered dataset from 4378 to 4243 examples with audio length <= 30 seconds


Map (num_proc=20):   0%|          | 0/4243 [00:00<?, ? examples/s]

Loaded TRAIN dataset with 4243 examples


In [ ]:
#train_dataset = load_dataset(DATASET_NAME, split='train', limit_to_30_seconds=True)
#train_dataset = train_dataset.map(
    #prepare_features, 
    #remove_columns=['audio'], 
    #writer_batch_size=1, 
    #num_proc=num_proc)
    
#print(f"TRAIN before filtering: {len(train_dataset)} examples")
#train_dataset = train_dataset.filter(is_valid_label_length)
#print(f"TRAIN after filtering: {len(train_dataset)} examples")

In [17]:
test_dataset = load_dataset(DATASET_NAME, split='test', limit_to_30_seconds=True)
test_dataset = test_dataset.map(prepare_features, remove_columns=['audio'], writer_batch_size=1, num_proc=num_proc)
print(f"Loaded TEST dataset with {len(test_dataset)} examples")

Filter:   0%|          | 0/928 [00:00<?, ? examples/s]

Filtered dataset from 928 to 926 examples with audio length <= 30 seconds


Map (num_proc=20):   0%|          | 0/926 [00:00<?, ? examples/s]

Loaded TEST dataset with 926 examples


In [ ]:
#test_dataset = load_dataset(DATASET_NAME, split='test', limit_to_30_seconds=True)
#test_dataset = test_dataset.map(
    #prepare_features, 
    #remove_columns=['audio'], 
    #writer_batch_size=1, 
    #num_proc=num_proc)
#print(f"TEST before filtering: {len(test_dataset)} examples")
#test_dataset = test_dataset.filter(is_valid_label_length)
#print(f"TEST after filtering: {len(test_dataset)} examples")

In [18]:
dev_dataset = load_dataset(DATASET_NAME, split='validation', limit_to_30_seconds=True)
dev_dataset = dev_dataset.map(prepare_features, remove_columns=['audio'], writer_batch_size=1, num_proc=num_proc)
print(f"Loaded DEV dataset with {len(dev_dataset)} examples")

Filter:   0%|          | 0/542 [00:00<?, ? examples/s]

Filtered dataset from 542 to 542 examples with audio length <= 30 seconds


Map (num_proc=20):   0%|          | 0/542 [00:00<?, ? examples/s]

Loaded DEV dataset with 542 examples


In [ ]:
# DEV/VALIDATION dataset
#dev_dataset = load_dataset(DATASET_NAME, split='validation', limit_to_30_seconds=True)
#dev_dataset = dev_dataset.map(
    #prepare_features, 
    #remove_columns=['audio'], 
    #writer_batch_size=1, 
    #num_proc=num_proc)
#print(f"DEV before filtering: {len(dev_dataset)} examples")
#dev_dataset = dev_dataset.filter(is_valid_label_length)
#print(f"DEV after filtering: {len(dev_dataset)} examples")

## Prepare Trainer

--> no need to change anything here, just run

Whenever something is changed in the settings, you need to rerun this part.

In [19]:
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_NAME)
_ = base_model.to(device)
print('Using Language: ', LANGUAGE)
print('Using model:', WHISPER_MODEL_TYPE)

# ensure task and language for training
base_model.generation_config.language = LANGUAGE
base_model.generation_config.task = TASK
base_model.generation_config.forced_decoder_ids = None
base_model.config.forced_decoder_ids = None
# to use gradient checkpointing
base_model.config.use_cache = False
print('language set to:', base_model.generation_config.language)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Using Language:  en
Using model: openai/whisper-small
language set to: en


In [20]:
# Add SpecAugment
if USE_SPECAUGMENT:
    base_model.config.apply_spec_augment = USE_SPECAUGMENT

    # Specaugment (use default settings, as per paper)

    # time masking
    base_model.config.mask_time_prob = 0.05
    base_model.config.mask_time_length = 10
    base_model.config.mask_time_min_masks = 2

    # feature masking
    base_model.config.mask_feature_prob = 0.05 # def: 0
    base_model.config.mask_feature_length = 10
    base_model.config.mask_feature_min_masks = 2 # def: 0

print('Using specaugment:', base_model.config.apply_spec_augment)


Using specaugment: True


In [21]:
# which layers to tune

print("Updating encoder:", UPDATE_ENCODER)
print("Updating projection layer:", UPDATE_PROJ)
print("Updating decoder:", UPDATE_DECODER)


base_model.model.encoder.requires_grad_(UPDATE_ENCODER)
base_model.model.decoder.requires_grad_(UPDATE_DECODER)
base_model.proj_out.requires_grad_(UPDATE_PROJ)

print("Overview to number of model parameters to be updated:")
print('* encoder params to update/total:', count_trainable_parameters(base_model.model.encoder), base_model.model.encoder.num_parameters())
print('* decoder parans to update/total:', count_trainable_parameters(base_model.model.decoder), base_model.model.decoder.num_parameters())

print('* overall # trainable parameters:', count_trainable_parameters(base_model))
print('*     overall # model parameters:', base_model.model.num_parameters())

Updating encoder: True
Updating projection layer: True
Updating decoder: True
Overview to number of model parameters to be updated:
* encoder params to update/total: 88154112 88154112
* decoder parans to update/total: 153580800 153580800
* overall # trainable parameters: 241734912
*     overall # model parameters: 241734912


In [22]:
# --- Partial Unfreeze ---

print("Updating encoder:", UPDATE_ENCODER)
print("Updating projection layer:", UPDATE_PROJ)
print("Updating decoder: PARTIAL (last 6 layers)")

# 1. Unfreeze Encoder and Projection Layer as usual
base_model.model.encoder.requires_grad_(UPDATE_ENCODER)
base_model.proj_out.requires_grad_(UPDATE_PROJ)

# 2. Freeze the ENTIRE decoder first
base_model.model.decoder.requires_grad_(False)

# 3. Manually unfreeze ONLY the last 6 decoder layers
# whisper-small has 12 layers (0-11). We unfreeze 6, 7, 8, 9, 10, 11.
num_layers_to_unfreeze = 2
for layer in base_model.model.decoder.layers[-num_layers_to_unfreeze:]:
    layer.requires_grad_(True)

print("\nOverview to number of model parameters to be updated:")
print(f'* encoder params to update/total: {count_trainable_parameters(base_model.model.encoder)} / {base_model.model.encoder.num_parameters()}')
print(f'* decoder params to update/total: {count_trainable_parameters(base_model.model.decoder)} / {base_model.model.decoder.num_parameters()}')
print(f'* overall # trainable parameters: {count_trainable_parameters(base_model)}')
print(f'* overall # model parameters: {base_model.model.num_parameters()}')

Updating encoder: True
Updating projection layer: True
Updating decoder: PARTIAL (last 6 layers)

Overview to number of model parameters to be updated:
* encoder params to update/total: 88154112 / 88154112
* decoder params to update/total: 18900480 / 153580800
* overall # trainable parameters: 107054592
* overall # model parameters: 241734912


In [ ]:
#!pip install --upgrade transformers accelerate datasets

In [23]:
# Training Hyper Parameters
# don't change settings here, but instead at very top!
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
    logging_steps=LOGGING_STEPS,
    report_to=["tensorboard"],
    include_num_input_tokens_seen=True,
    ### on GPU, can either do fp16 or bf16 depending on specific GPU
    fp16=USE_FP16, 
    bf16=USE_BF16, 
    push_to_hub=False,
    remove_unused_columns=False,
    #
    num_train_epochs=MAX_EPOCHS,
    max_steps=MAX_STEPS,
    #
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    #
    per_device_train_batch_size=BATCH_SIZE,
    #
    eval_on_start=EVAL_ON_START,
    predict_with_generate=True,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    generation_max_length=MAX_GEN_LEN,
    #
    metric_for_best_model="wer",
    greater_is_better=False,
    #
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    
    #
    # only applies to polynomial schedule (constant ignores args)
    lr_scheduler_kwargs={
        "lr_end": LR_END, # The final LR.  Crucial for polynomial decay.
        "power": LR_DECAY_POWER, # for decay
        # we don't need to set the other arguments as they are already set in the args outside
        #"num_warmup_steps": WARMUP_STEPS, # The number of steps for the warmup phase.
        #"num_training_steps": MAX_STEPS, # The total number of training steps.
        #"lr_init": 1e-5 # we take the LR setting
    },

    learning_rate=LEARNING_RATE,
    warmup_steps=LR_WARMUP_STEPS, # what happens if we have this and the LR schedule args ?
    #
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=NUM_CHECKPOINTS_TO_STORE,
    load_best_model_at_end=True,
    # group_by_length=True
    # auto_find_batch_size=True
)

print('trainer args set, writing to:', OUTPUT_DIR)

trainer args set, writing to: /jupyter_kernel/trained_models/whisper-small-kenyan-english-nonstandard_v1_run1


## Weight Decay & Early stopping

In [24]:
print("--- MODIFYING TRAINING ARGS ---")

# 1. Add Weight Decay
# This is the key to fighting the overfitting.
training_args.weight_decay = 0.01

# 2. Add Early Stopping
# This will stop training when the WER stops improving.
training_args.early_stopping_patience = 3

print(f"Set weight_decay to: {training_args.weight_decay}")
print(f"Set early_stopping_patience to: {training_args.early_stopping_patience}")
print("---------------------------------")

--- MODIFYING TRAINING ARGS ---
Set weight_decay to: 0.01
Set early_stopping_patience to: 3
---------------------------------


In [25]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=base_model.config.decoder_start_token_id,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor
)


[RANK 0] Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## Run the training

Note: tensorboard doesn't show properly in jupyter notebooks, use the tensorboard_server.py tool to host a tensorboard instance on Modal, using below model training dir:

In [26]:
print('model training dir:', OUTPUT_DIR)

model training dir: /jupyter_kernel/trained_models/whisper-small-kenyan-english-nonstandard_v1_run1


In [ ]:
#import torch
#import gc

# Clear all cached memory
# torch.cuda.empty_cache()
# gc.collect()

# Set memory allocation strategy
# import os
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# train from scratch
# trainer.train()

# # alternatively, you can continue training if a previous job was interrupted
trainer.train(resume_from_checkpoint = True)


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Step,Training Loss,Validation Loss,Wer,Cer,Input Tokens Seen
400,1.048900,0.958325,0.254019,0.159536,3062640000
450,0.973900,0.945866,0.255973,0.162220,3446640000
500,0.962700,0.933803,0.253913,0.161757,3830640000
550,0.774000,0.928145,0.248399,0.157283,4211520000
600,0.873300,0.920946,0.248810,0.157320,4595520000
650,1.029200,0.911163,0.246974,0.155844,4979520000
700,0.846400,0.908627,0.244899,0.154962,5360400000
750,0.880800,0.902853,0.243267,0.152940,5744400000
800,0.844600,0.900214,0.244314,0.154957,6125280000
850,0.775700,0.896716,0.244043,0.156342,6509280000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

adjusted: 0.25401882226950273 0.15953646837991484
un-adjusted: 0.29630980121869205 0.18204954797138995


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

...

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

adjusted: 0.2388320337277462 0.1541161959010625
un-adjusted: 0.3188362140380913 0.19624632408190004


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

adjusted: 0.23943083558061445 0.15501080601850345
un-adjusted: 0.32685252484865784 0.2021327607362811


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

### On DEV set

In [ ]:
# (should give the same result shown in trainig progress on dev set)
trainer.evaluate(dev_dataset, language=LANGUAGE)

### On TEST set

In [ ]:
# run on dev-set 
# (should give the same result shown in trainig progress on dev set)
trainer.evaluate(test_dataset, language=LANGUAGE)

## Store Model

--> save best model

### Save to your volume

In [ ]:
# with "load_best_model_at_end=True" set in the settings (this is the default, so don't change that), after training is completed the best model is loaded and then saved
best_model_dir = os.path.join(OUTPUT_DIR, 'best_model')
print(f"Saving to: {best_model_dir}")
trainer.model.save_pretrained(best_model_dir, safe_serialization=True)
trainer.tokenizer.save_pretrained(best_model_dir)